# Spaghetti
Make spaghetti plots for ENSO events

## Imports

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import pandas as pd

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_spaghetti(idx, data, peak_month, event_type=None, q=0.95, is_warm=True):
    """
    Get hovmoller composite based on specified:
    - data: used to compute index/make composite
    - peak_month: month to center composite on
    - q: quantile threshold for composite
    """

    ## handle warm/cold case
    if is_warm:
        kwargs = dict(q=q, check_cutoff=lambda x, cut: x > cut)
    else:
        kwargs = dict(q=1 - q, check_cutoff=lambda x, cut: x < cut)

    ## kwargs for composite
    kwargs = dict(
        kwargs,
        avg=False,
        peak_month=peak_month,
        idx=idx,
        data=data,
        event_type=event_type,
    )

    ## composite of projected data
    spag = src.utils.make_composite(**kwargs)

    return spag


def save(fig, fname, dpi=300, do_save=False):
    """save figure to file"""

    ## get save directory
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "ch3-outline")

    ## get fname
    fname = save_dir / f"{fname}.pdf"

    if do_save:
        fig.savefig(fname, dpi=dpi)

    return


def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom


def load_h_ohc_idxs():
    """load OHC-based h indices"""

    ## specify subsetting funcs
    LONS_E = dict(longitude=slice(210, 270))
    LONS_W = dict(longitude=slice(120, 180))

    ## averaging funcs
    lon_avg = lambda x, lons: x.sel(lons).mean("longitude")
    lat_avg = lambda x: x.sel(latitude=slice(-5, 5)).mean("latitude")

    ## load spatial data
    _, anom_wide = load_consolidated_wide()

    ## empty array to hold result
    h_idxs = xr.Dataset()

    ## compute ohc
    h_idxs["h_w_ohc"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_W) / 300,
    )["T"]
    h_idxs["h_e_ohc"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_E) / 300,
    )["T"]

    h_idxs["h_ohc"] = 0.5 * (h_idxs["h_w_ohc"] + h_idxs["h_e_ohc"])
    h_idxs["dh_ohc"] = h_idxs["h_e_ohc"] - h_idxs["h_w_ohc"]

    return h_idxs

## Load data

### T, h

In [ ]:
## fits

## Th data
Th = src.utils.load_cesm_indices(load_z20=True)

## load ohc data
Th = xr.merge([Th, load_h_ohc_idxs()])

## omit first year (bc of NaN in h,hw vars)
Th = Th.sel(time=slice("1851", None))

#### Load heat flux

In [ ]:
_, anom = src.utils.load_consolidated()

In [ ]:
## get NHF in Niño regions
nhf_3 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino3
)["nhf"]
nhf_34 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino34
)["nhf"]

## convert to units of K/mo
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H = 50

Q = nhf_34 * sec_per_yr / (rho * Cp * H)
Q_3 = nhf_3 * sec_per_yr / (rho * Cp * H)

## merge with Th data
Th["Q"] = Q.sel(time=Th.time)
Th["Q_3"] = Q_3.sel(time=Th.time)

#### get windowed data

In [ ]:
## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)

### Early and late

In [ ]:
## specify years
years = [2011, 2081]

## get early and late
Th_early = Th_rolling.sel(year=years[0])
Th_late = Th_rolling.sel(year=years[1])

## remove median
Th_early = Th_early.groupby("time.month") - Th_early.groupby("time.month").median()
Th_late = Th_late.groupby("time.month") - Th_late.groupby("time.month").median()

## compute spaghetti/composite

In [ ]:
## specify composite specs
comp_kwargs = dict(
    is_warm=True,
    event_type=None,
    peak_month=12,
    q=0.95,
)

## specify variable to use for composite
VARNAME = "T_34"

## do the compute
spag_early = get_spaghetti(
    idx=Th_early[VARNAME], data=Th_early, **comp_kwargs
).drop_vars("year")
spag_late = get_spaghetti(idx=Th_late[VARNAME], data=Th_late, **comp_kwargs).drop_vars(
    "year"
)

## plots

### Mean

Clean up; add bounds

In [ ]:
def prep(spag):
    """prep spaghetti"""

    ## differentiate / normalize
    spag["ddt_T"] = 12 * spag["T_34"].differentiate("lag")
    spag["Q_gr"] = spag["Q"] / spag["T_34"]
    spag["gr"] = spag["ddt_T"] / spag["T_34"]

    ## rolling
    return spag.rolling({"lag": 3}, center=True).mean()
    # return spag


def relu_neg(x):
    """get (negative) relu of dataarray"""
    return x.where(x < 0, other=0)


spag_diff = prep(spag_late) - prep(spag_early)
spag_diff_q = spag_diff.quantile(q=[0.1, 0.5, 0.9], dim="sample").rename(
    {"quantile": "q"}
)

In [ ]:
## specify y-lim
xlim = [-7.2, 3]
ylim = [-6, 6]

## specify colors
colors = sns.color_palette()

fig, axs = plt.subplots(1, 3, figsize=(5.5, 2.5), layout="constrained")

## plot difference
for i, (sp, c, yr) in enumerate(zip([spag_early, spag_late], colors, [2010, 2080])):
    sp_ = (
        prep(sp)["T_34"]
        .quantile(q=[0.1, 0.5, 0.9], dim="sample")
        .rename({"quantile": "q"})
    )
    # for q, lw in zip(sp_.q, [.7,4,.7]):
    axs[0].plot(sp_.sel(q=0.5), sp.lag, lw=4, c=c, label=yr)

    ## plot bounds
    lb, ub = [sp_.sel(q=q) for q in [0.1, 0.9]]
    if i:
        for q in sp_.q:
            axs[0].plot(sp_.sel(q=q), sp.lag, lw=1, c=c)
    else:
        axs[0].fill_betweenx(y=sp_.lag, x1=lb, x2=ub, color=c, alpha=0.1)

for (
    ax,
    v,
) in zip(axs[1:], ["ddt_T", "Q"]):
    for q, lw in zip([0.5, 0.1, 0.9], [4, 1, 1]):
        ax.plot(spag_diff_q[v].sel(q=q), spag_diff_q.lag, lw=lw, c=colors[4])

    ## highlight damping
    med = spag_diff_q[v].sel(q=0.5)
    ax.fill_betweenx(
        y=med.lag,
        x2=relu_neg(med),
        x1=np.zeros_like(med.lag),
        color="k",
        alpha=0.1,
        lw=0,
    )
    ax.set_yticks([])
    ax.set_xlabel(r"K yr$^{-1}$")
    ax.set_xticks([-4, 0])
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

## formatting
# axs[0].legend(prop=dict(size=10))
axs[0].set_xlabel(r"K")
# axs[0].set_ylabel("Lag (months)")
axs[0].set_xticks([0, 3])
axs[0].set_yticks([-6, -2, 1, 6], labels=["Jun(-1)", "Oct(-1)", "Jan(+1)", "Jun(+1)"])
axs[0].set_ylim(ylim)
axs[0].set_xlim([-0.5, None])
f_kwargs = dict(lw=0.75, c="k", ls="--")
for ax in axs:
    ax.axvline(0, c="k", lw=1)
    ax.axhline(1, **f_kwargs)
    ax.axhline(-2, **f_kwargs)

axs[0].text(s=2010, color=colors[0], x=3, y=-5)
axs[0].text(s=2080, color=colors[1], x=0.5, y=-1)

## add titles
axs[0].set_title(r"(c) $T_{34}$", loc="left")
axs[1].set_title(r"(d) $\Delta\left(\frac{d T_{34}}{dt}\right)$", loc="left")
axs[2].set_title(r"(e) $\Delta ~Q_{34}$", loc="left")

plt.show()

## Get $R$ over time

### Nov-Jan

In [ ]:
show_all=False
name="pos"
(not(show_all)) & (name=="all")

In [ ]:
def regress_over_time(
    data_windowed,
    x_vars,
    y_vars,
    dims=["time"],
    by_month=False,
):
    """regression over time"""

    ## shared args
    kwargs = dict(x_vars=x_vars, y_vars=y_vars, dims=dims)

    ## do regression
    if by_month:
        coefs = data_windowed.groupby("time.month").map(src.utils.regress_xr_proj, **kwargs)
    else:
        coefs = src.utils.regress_xr_proj(data_windowed, **kwargs)

    return coefs.drop_vars("j")


def plot_timeseries_v2(ax, coefs, center=False, show_all=False):
    """plot timeseries comparison"""

    ## normalize coefficients
    coefs_center = coefs-coefs.isel(year=0).mean("member")

    ## select coefficients to plot
    if center:
        plot_coefs = coefs_center
    else:
        plot_coefs = coefs

    ## loop thru pos and negative
    for i, (name, color, label) in enumerate(
        zip(["pos", "neg", "all"], ["r", "b", "k"], [r"$T_{34}>0$", "$T_{34}<0$","all"])
    ):
        # if False:
        if ((not(show_all)) & (name=="all")):
            pass

        else:
            
            ## plot median and bounds
            if name == "all":
                q_vals = [0.5]
            else:
                q_vals = [0.5, 0.1, 0.9]
        
            for q, lw in zip(q_vals, [2, 0.6, 0.6]):
                label_ = label if (q == 0.5) else None
        
                ## finally, plot data
                ax.plot(
                    plot_coefs.year,
                    plot_coefs[name].quantile(q=q, dim="member"),
                    c=color,
                    lw=lw,
                    label=label_,
                )

    ## formatting
    ax.set_xticks([1870, 2010])
    ax_kwargs = dict(ls="--", c="gray", lw=0.8)
    ax.axvline(2010, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)
    # ax.set_yticks([])

    return fig, axs


def plot_R_comparison(axs, R, alpha, **kwargs):
    """plot comparison of R stuff on axs"""

    ## Bjerknes
    plot_timeseries_v2(axs[0], coefs=R, **kwargs)

    ## damping
    plot_timeseries_v2(axs[1], coefs=alpha, **kwargs)

    ## difference
    plot_timeseries_v2(axs[2], coefs=R - alpha, **kwargs)

    ##formatting
    for ax in axs[1:]:
        ax.set_yticks([])
    axs[0].set_ylabel(r"yr$^{-1}$")

    return axs

In [ ]:
T_VAR, Q_VAR = "T_34", "Q"
# T_VAR, Q_VAR = "T_3", "Q_3"
data_ = Th_rolling[[T_VAR, Q_VAR]]
T_idx = data_[T_VAR]

#### Compute $\alpha$

In [ ]:
## select months
# MONTHS_ALPHA = [10,11,12]
# MONTHS_R = [10,1]
MONTHS_ALPHA = [11, 12, 1]
MONTHS_R = [11, 2]

## get data to compute alpha
data_ndj = src.utils.sel_month(data_, months=MONTHS_ALPHA)
data_pos = data_ndj.where(T_idx > 0)
data_neg = data_ndj.where(T_idx < 0)

## compute
kwargs = dict(x_vars=[T_VAR], y_vars=[Q_VAR])
alpha_all = regress_over_time(data_ndj, **kwargs)[Q_VAR]
alpha_pos = regress_over_time(data_pos, **kwargs)[Q_VAR]
alpha_neg = regress_over_time(data_neg, **kwargs)[Q_VAR]
alpha = xr.merge(
    [alpha_all.rename("all"), alpha_pos.rename("pos"), alpha_neg.rename("neg")]
)

## flip sign
alpha *= -1

#### Compute $R$

In [ ]:
## get data to compute alpha
data0 = src.utils.sel_month(data_[T_VAR], months=MONTHS_R[0]).isel(
    time=slice(None, -1)
)
data1 = src.utils.sel_month(data_[T_VAR], months=MONTHS_R[1]).isel(time=slice(1, None))
data1 = data1.assign_coords({"time": data0.time})
data_all = xr.merge([data0.rename("T0"), data1.rename("Tf")])
data_pos = data_all.where(data_all["T0"] > 0)
data_neg = data_all.where(data_all["T0"] < 0)

## compute
kwargs = dict(x_vars=["T0"], y_vars=["Tf"])
S_all = regress_over_time(data_all, **kwargs)["Tf"]
S_pos = regress_over_time(data_pos, **kwargs)["Tf"]
S_neg = regress_over_time(data_neg, **kwargs)["Tf"]
S = xr.merge([S_all.rename("all"), S_pos.rename("pos"), S_neg.rename("neg")])

## divide by 3 to convert to 1/month, then mult. by 12 to convert to 1/yr
tau = 3  # months
mpy = 12  # months per year
R = (1 / tau * mpy) * np.log(S)

#### Plot

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(8, 2.75), layout="constrained")

axs = plot_R_comparison(axs, R=R, alpha=-alpha)

## formatting
labels = [r"$R$", r"$\alpha$", r"$R_o=R+\alpha$"]
labels = [r"a) $T_{34}$ growth rate", r"b) $Q_{34}$ contribution", r"c) Residual (a-b)"]
for ax, label in zip(axs, labels):
    ax.set_title(label, loc="left")

## formatting
yticks = np.array([-3, 0, 1])
ylim = np.array([-3.5, 1.5])
axs[0].set_yticks(yticks)
axs[2].set_yticks((-yticks)[::-1])
axs[0].set_ylim(ylim)
for ax in axs[2:]:
    ax.set_ylim((-ylim)[::-1])

src.utils.set_lims(axs[:2])
axs[0].legend()

plt.show()

### All months

#### Compute $\alpha$

In [ ]:
data_pos = data_.where(T_idx > 0)
data_neg = data_.where(T_idx < 0)

## compute
kwargs = dict(x_vars=[T_VAR], y_vars=[Q_VAR], by_month=True)
alpha_all = regress_over_time(data_, **kwargs)[Q_VAR]
alpha_pos = regress_over_time(data_pos, **kwargs)[Q_VAR]
alpha_neg = regress_over_time(data_neg, **kwargs)[Q_VAR]
alpha = xr.merge(
    [alpha_all.rename("all"), alpha_pos.rename("pos"), alpha_neg.rename("neg")]
)

## flip sign
alpha *= -1

#### Compute $R$

In [ ]:
## get data to compute alpha
data0 = data_[T_VAR].isel(time=slice(None, -1))
data1 = data_[T_VAR].isel(time=slice(1, None))
data1 = data1.assign_coords({"time": data0.time})
data_all = xr.merge([data0.rename("T0"), data1.rename("Tf")])
data_pos = data_all.where(data_all["T0"] > 0)
data_neg = data_all.where(data_all["T0"] < 0)

## compute
kwargs = dict(x_vars=["T0"], y_vars=["Tf"], by_month=True)
S_all = regress_over_time(data_all, **kwargs)["Tf"]
S_pos = regress_over_time(data_pos, **kwargs)["Tf"]
S_neg = regress_over_time(data_neg, **kwargs)["Tf"]
S = xr.merge([S_all.rename("all"), S_pos.rename("pos"), S_neg.rename("neg")])

## divide by 3 to convert to 1/month, then mult. by 12 to convert to 1/yr
tau = 1  # months
mpy = 12  # months per year
R = (1 / tau * mpy) * np.log(S)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(8, 2.75), layout="constrained")

# sel_ndj = lambda x : x.sel(month=[11,12,1]).mean("month")
# sel_ndj = lambda x : x.sel(month=slice(6,8)).mean("month")
sel_ndj = lambda x : x.mean("month")

axs = plot_R_comparison(axs, R=sel_ndj(R), alpha=-sel_ndj(alpha), show_all=True,center=True)

## formatting
labels = [
    r"a) $\Delta T_{34}$ growth rate", 
    r"b) $\Delta Q_{34}$ contribution", 
    r"c) Residual (a-b)"
]
for ax, label in zip(axs, labels):
    ax.set_title(label, loc="left")

## formatting
# axs[0].set_ylim([-.5,.5])
# axs[0].set_yticks([-.5,0,.5])
for ax in axs:
    ax.set_ylim([-1,1])
axs[0].set_yticks([-1,0,1])

axs[0].legend()

plt.show()

Plot seasonal cycle

In [ ]:
months = np.arange(1,13)
fig,axs = plt.subplots(1,3,figsize=(8,2.5))
for ax, n in zip(axs, ["all","pos","neg"]):
    for y in [1871, 2011, 2051, 2081]:
        ax.plot(months, (R)[n].mean("member").sel(year=y))
    ax.axhline(0, ls="--", c="k", lw=1)
    ax.set_yticks([])
    ax.set_xticks([2,7, 12], labels=["Feb","Jul", "Dec"])
    ax.set_title(n)

src.utils.set_lims(axs)
axs[0].set_yticks([-4,0,4])
plt.show()